In [2]:
import numpy as np
import pandas as pd

# LOAD + PREP DATA

df = pd.read_csv("Task 3 and 4_Loan_Data.csv")
df = df[["fico_score", "default"]].dropna()
df = df.sort_values("fico_score").reset_index(drop=True)


# AGGREGATE BY FICO SCORE

fico_agg = (
    df.groupby("fico_score")
      .agg(
          defaults=("default", "sum"),
          count=("default", "size")
      )
      .reset_index()
)

defaults = fico_agg["defaults"].values
counts = fico_agg["count"].values
fico_vals = fico_agg["fico_score"].values

N = len(fico_agg)


# LOG-LIKELIHOOD FUNCTION

def bucket_log_likelihood(k, n):
    if k == 0 or k == n:
        return 0.0
    p = k / n
    return k * np.log(p) + (n - k) * np.log(1 - p)


# BUILD LIKELIHOOD MATRIX

ll = np.zeros((N, N))

for i in range(N):
    k = 0
    n = 0
    for j in range(i, N):
        k += defaults[j]
        n += counts[j]
        ll[i, j] = bucket_log_likelihood(k, n)

# DYNAMIC PROGRAMMING

def optimal_fico_buckets_dp(ll, n_buckets):
    N = ll.shape[0]
    dp = np.full((N, n_buckets), -np.inf)
    split = np.zeros((N, n_buckets), dtype=int)

    for i in range(N):
        dp[i, 0] = ll[0, i]

    for b in range(1, n_buckets):
        for i in range(b, N):
            for j in range(b-1, i):
                val = dp[j, b-1] + ll[j+1, i]
                if val > dp[i, b]:
                    dp[i, b] = val
                    split[i, b] = j

    boundaries = []
    idx = N - 1
    for b in reversed(range(n_buckets)):
        boundaries.append(idx)
        idx = split[idx, b]

    return boundaries[::-1]


# BUILD RATING MAP

def build_rating_map_from_agg(agg_df, boundaries):
    buckets = []
    start = 0

    for i, end in enumerate(boundaries):
        bucket = agg_df.iloc[start:end+1]
        buckets.append({
            "rating": i + 1,  # 1 = best credit
            "fico_min": bucket["fico_score"].min(),
            "fico_max": bucket["fico_score"].max(),
            "pd": bucket["defaults"].sum() / bucket["count"].sum(),
            "count": bucket["count"].sum()
        })
        start = end + 1

    return pd.DataFrame(buckets)


# RUN BUCKETING

N_BUCKETS = 5
boundaries = optimal_fico_buckets_dp(ll, N_BUCKETS)
rating_map = build_rating_map_from_agg(fico_agg, boundaries)

rating_map


,rating,fico_min,fico_max,pd,count
0,1,408,520,0.661130,301
1,2,521,580,0.380952,1407
2,3,581,640,0.204479,3438
3,4,641,696,0.105099,3197
4,5,697,850,0.046470,1657
